# 05 — cuda c++ basics

companion workspace for NVIDIA's [intro to CUDA C++](https://docs.nvidia.com/cuda/cuda-programming-guide/02-basics/intro-to-cuda-cpp.html), followed through the [Wafer curriculum](../resources/README.md).

select **lmlab's `.venv` Python kernel**. Python runs the compiler and executable; CUDA C++ stays in ordinary editable `.cu` files. start with the [existing smoke program](../src/gpu_primitives/smoke_cuda.cu), then save your implementations in `src/gpu_primitives/` and change `source_file` below.

this notebook is prepared but unexecuted. it supplies build/run plumbing, not completed learning exercises. running a program uses your GPU; compiling does not run it.

## workspace and editable settings

run these setup cells first. build outputs stay in the ignored `.build/` directory. change `program_args` to match the arguments accepted by your own program; the existing smoke program does not use them.

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

repo_root = next(
    (path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (path / "src/gpu_primitives").is_dir() and (path / "pyproject.toml").is_file()),
    None,
)
if repo_root is None:
    raise RuntimeError("open this notebook from the lmlab workspace")

cuda_home = Path(os.environ.get("CUDA_HOME", "/usr/local/cuda-13.3"))
cuda_arch = os.environ.get("CUDA_ARCH", "sm_86")  # this machine's RTX 3080
source_file = repo_root / "src/gpu_primitives/smoke_cuda.cu"
program_args = []  # e.g. ["1025"] once your program accepts a vector length
build_dir = repo_root / ".build/05_cuda_cpp_basics"

print(f"workspace: {repo_root}")
print(f"editable source: {source_file}")
print(f"target: {cuda_arch}")

In [ ]:
def cuda_tool(name):
    candidate = cuda_home / "bin" / name
    if candidate.is_file():
        return str(candidate)
    available = shutil.which(name)
    if available is None:
        raise FileNotFoundError(f"{name} missing; check CUDA_HOME and LOCAL_SETUP.md")
    return available


def run_command(arguments):
    # Stream diagnostics, including compiler errors, directly into cell output.
    subprocess.run([str(arg) for arg in arguments], cwd=repo_root, check=True)


def build_cuda(source):
    source = Path(source)
    if not source.is_file() or source.suffix != ".cu":
        raise FileNotFoundError(f"select an existing editable .cu file: {source}")
    build_dir.mkdir(parents=True, exist_ok=True)
    executable = build_dir / source.stem
    # Remove stale output so a failed build cannot leave an old binary to run.
    executable.unlink(missing_ok=True)
    run_command([
        cuda_tool("nvcc"), "-std=c++17", f"-arch={cuda_arch}", "-lineinfo",
        source, "-o", executable,
    ])
    return executable

## 1. compilation with nvcc

read [§2.1.1](https://docs.nvidia.com/cuda/cuda-programming-guide/02-basics/intro-to-cuda-cpp.html#compilation-with-nvcc). compile the selected file below. after editing C++, rebuild before running. a fresh executable starts a fresh C++ process; C++ variables do not persist between Python cells.

In [ ]:
run_command([cuda_tool("nvcc"), "--version"])
executable = build_cuda(source_file)

## 2. kernels, launches, and indexing

read [§2.1.2](https://docs.nvidia.com/cuda/cuda-programming-guide/02-basics/intro-to-cuda-cpp.html#kernels). connect `__global__`, launch dimensions, and thread indices to the elements your kernel handles. include a bounds check when your input does not fill the last block.

edit your source, rebuild, then run:

In [ ]:
run_command([executable, *program_args])

### working notes

- source file:
- launch configuration:
- what each thread handles:
- observed result:
- next change:

## 3. unified memory

read [§2.1.3.1](https://docs.nvidia.com/cuda/cuda-programming-guide/02-basics/intro-to-cuda-cpp.html#unified-memory). write your managed-memory version in a new `.cu` file. select it with `source_file`, then reuse the build and run cells. keep the host reference result and a numerical comparison visible in your program's output.

### working notes

- allocation and ownership:
- when the host reads the result:
- correctness evidence:
- questions:

## 4. explicit memory management

read [§2.1.3.2](https://docs.nvidia.com/cuda/cuda-programming-guide/02-basics/intro-to-cuda-cpp.html#explicit-memory-management). implement the explicit-memory variant in another file. trace host allocation, device allocation, transfers, computation, result retrieval, and cleanup. reuse the same input and correctness comparison.

### working notes

- source file:
- transfer directions and sizes:
- comparison with the managed-memory version:
- questions:

## 5. synchronization and errors

read [§2.1.4](https://docs.nvidia.com/cuda/cuda-programming-guide/02-basics/intro-to-cuda-cpp.html#synchronizing-cpu-and-gpu). a kernel launch can return before GPU work finishes. make the host wait before consuming results. check CUDA return values and distinguish launch errors from execution errors.

the following optional cell runs the selected executable under Compute Sanitizer. zero reported memory errors complements numerical checking; it does not establish numerical correctness by itself.

In [ ]:
run_command([
    cuda_tool("compute-sanitizer"), "--tool", "memcheck", "--error-exitcode", "1",
    executable, *program_args,
])

## 6. putting it together

use [§2.1.5](https://docs.nvidia.com/cuda/cuda-programming-guide/02-basics/intro-to-cuda-cpp.html#putting-it-all-together) as a reference while completing your own programs. once they accept vector length as an argument, try lengths on both sides of a block boundary.

record evidence below. no timing results are supplied; timing compilation or launching a process would not isolate kernel execution.

| source | vector length | threads per block | numerical check | sanitizer | observation |
|---|---|---|---|---|---|
| | | | | | |

## next question

what changed your understanding, and what will you inspect next?